# Linear Regression

## Load the Dataset

In [12]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-topic-modeling.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-topic-modeling.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-topic-modeling.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,first_place,...,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf,topic_bertopic
0,0,2014-11-20 06:43:16.005,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3052,150,0.049148,True,...,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177,-1
1,1,2014-11-20 06:43:44.646,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3033,122,0.040224,False,...,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177,-1
2,2,2014-11-20 06:44:59.804,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3092,110,0.035576,False,...,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177,-1
3,3,2014-11-20 06:54:36.335,201446,546d902c26714c6c44000039,This Is What Sexism Against Men Sounds Like,546bc55335992b86c8000043,3526,90,0.025525,False,...,0.737,0.263,0.3612,3,6,8,82.390000,6.6,0.486990,-1
4,4,2014-11-20 06:54:57.878,201446,546d902c26714c6c44000039,This Is What Sexism Against Men Sounds Like,546d900426714cd2dd00002e,3506,120,0.034227,True,...,0.737,0.263,0.3612,3,6,8,82.390000,6.6,0.486990,-1


### Basic Linear Regression

First I'll run a basic linear regression model to predict the click rate from the features. All the numerical features will be used as predictors.

This will be affected by severe multicollinearity but it's a good starting point.

In [13]:
exploratory_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22666 entries, 0 to 22665
Data columns (total 50 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   package_id                     22666 non-null  int64  
 1   created_at                     22666 non-null  object 
 2   test_week                      22666 non-null  int64  
 3   clickability_test_id           22666 non-null  object 
 4   headline                       22666 non-null  object 
 5   eyecatcher_id                  22644 non-null  object 
 6   impressions                    22666 non-null  int64  
 7   clicks                         22666 non-null  int64  
 8   ctr                            22666 non-null  float64
 9   first_place                    22666 non-null  bool   
 10  winner                         22666 non-null  bool   
 11  is_highest_ctr                 22666 non-null  bool   
 12  headline_num_persons           22666 non-null 

In [14]:
def prepare_data(df):
    # Transform int columns to float
    df["headline_num_persons"] = df["headline_num_persons"].astype(float)
    df["headline_num_orgs"] = df["headline_num_orgs"].astype(float)
    df["headline_num_gpes"] = df["headline_num_gpes"].astype(float)
    df["num_pronouns"] = df["num_pronouns"].astype(float)
    df["num_chars"] = df["num_chars"].astype(float)
    df["num_tokens"] = df["num_tokens"].astype(float)
    df["num_nouns"] = df["num_nouns"].astype(float)
    df["num_verbs"] = df["num_verbs"].astype(float)
    df["num_adjs"] = df["num_adjs"].astype(float)
    df["num_advs"] = df["num_advs"].astype(float)
    df["test_group_size"] = df["test_group_size"].astype(float)

    # Drop rows with missing values in any column
    df = df.dropna()
    return df

exploratory_df = prepare_data(exploratory_df)
confirmatory_df = prepare_data(confirmatory_df)
holdout_df = prepare_data(holdout_df)


In [16]:
exploratory_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22644 entries, 0 to 22665
Data columns (total 50 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   package_id                     22644 non-null  int64  
 1   created_at                     22644 non-null  object 
 2   test_week                      22644 non-null  int64  
 3   clickability_test_id           22644 non-null  object 
 4   headline                       22644 non-null  object 
 5   eyecatcher_id                  22644 non-null  object 
 6   impressions                    22644 non-null  int64  
 7   clicks                         22644 non-null  int64  
 8   ctr                            22644 non-null  float64
 9   first_place                    22644 non-null  bool   
 10  winner                         22644 non-null  bool   
 11  is_highest_ctr                 22644 non-null  bool   
 12  headline_num_persons           22644 non-null  floa

I'll use Python Statsmodels library

In [18]:
import statsmodels.api as sm

def basic_linear_regression(df):
    target_col = "ctr"
    feature_cols = [
        "headline_num_persons",
        "headline_num_orgs",
        "headline_num_gpes",
        "headline_has_person",
        "headline_has_org",
        "headline_has_gpe",
        "headline_has_money",
        "starts_with_verb",
        "starts_with_pronoun",
        "starts_with_number",
        "num_pronouns",
        "num_chars",
        "num_tokens",
        "avg_token_len",
        "ends_with_qmark",
        "ends_with_exclaim",
        "has_quote",
        "has_all_caps_word",
        "num_nouns",
        "num_verbs",
        "num_adjs",
        "num_advs",
        "headline_imperative_verb",
        "headline_has_curiosity_word",
        "headline_curiosity_similarity",
        "headline_has_intensity_word",
        "headline_intensity_similarity",
        "neg",
        "neu",
        "pos",
        "compound",
        "test_group_size",
        "read_flesch",
        "read_coleman",
        "specificity_tfidf",
    ]

    # Drop rows with missing values in any of these columns
    model_df = df[feature_cols + [target_col]].dropna()

    X = model_df[feature_cols]
    y = model_df[target_col]

    # Ensure all predictors and target are numeric (convert bools to 0/1 and ints to float)
    X = X.astype(float)
    y = y.astype(float)

    # Add intercept term
    X = sm.add_constant(X)

    # Fit OLS regression using Statsmodels
    return sm.OLS(y, X).fit()

ols_model = basic_linear_regression(exploratory_df)
# Display regression summary
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                    ctr   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.052
Method:                 Least Squares   F-statistic:                     36.24
Date:                Sat, 06 Dec 2025   Prob (F-statistic):          8.30e-237
Time:                        23:46:05   Log-Likelihood:                 68102.
No. Observations:               22644   AIC:                        -1.361e+05
Df Residuals:                   22608   BIC:                        -1.358e+05
Df Model:                          35                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

R-squared is very low, which means that only 5% of the variance is explained by the linear regression. This is expected, we don't think a linear regression over the engineered features will help us to certainly predict the click rate, but it's valueable to see which features are more important and could give us some insights.

The features that seem more significant (lower p-values and a confidence interval that doesn't include 0) are:
- headline_num_persons (0.007) -> People with names is mentioned in the headline, increases the click rate
- headline_has_money (0.001) -> Headlines that mention money are less effective
- num_chars (0.083) and num_tokens (0.005) -> Slightly decrease the click rate
- ends_with_qmark and ends_with_exclaim (0.000) -> Decrease the click rate
- num_nouns (0.000) -> Increases the click rate
- compound (0.000) -> Decreases the click rate, which means more positive sentiment in the headline decreases the click rate
- test_group_size (0.000) -> Larger test groups have lower click rates, this may be expected because, with larger test groups, headlines with poorer performance are more likely to be shown
- specificity_tfidf (0.000) -> More specificity decreases the click rate, which means that headlines with more unique words are less effective


This tell us with a minimal statistical significance that audience prefer to click on headlines that:
- Mention people by name and don't talk about money
- Are shorter rather that larger
- Aren't a question or an exclamation
- Have more nouns (This may be related to the mention of people)
- Have a more neutral or negative sentiment
- Have less specific words

All this makes sense, people prefers simpler headlines, more negative, and it's better if they talk about someone specifically.




## Analyze individual features

Now we'll run linear regressions to analyze the effect of specific changes in the headlines, and avoid the multicollinearity issue of fitting all the features at once.

These are the groups of features that will be fitted:

- Named entities: headline_num_persons, headline_num_orgs, headline_num_gpes
- Named entities boolean: headline_has_person, headline_has_org, headline_has_gpe, headline_has_money
- Sentence starting with: starts_with_verb, starts_with_pronoun, starts_with_number
- Basic text features: num_pronouns, num_nouns, num_verbs, num_adjs, num_advs, num_chars, num_tokens, avg_token_len
- Question/Exclamation features: ends_with_qmark, ends_with_exclaim, has_quote, has_all_caps_word",
- Sentence style: headline_imperative_verb, headline_has_curiosity_word, headline_curiosity_similarity, headline_has_intensity_word, headline_intensity_similarity
- Sentiment: neg, neu, pos, compound
- Test size: test_group_size
- Readability and specificity: read_flesch, read_coleman, specificity_tfidf

In [19]:
def feature_groups_regression(df):

    target_col = "ctr"

    feature_groups = {
        "named_entities": [
            "headline_num_persons",
            "headline_num_orgs",
            "headline_num_gpes",
        ],
        "named_entities_bool": [
            "headline_has_person",
            "headline_has_org",
            "headline_has_gpe",
            "headline_has_money",
        ],
        "sentence_start": [
            "starts_with_verb",
            "starts_with_pronoun",
            "starts_with_number",
        ],
        "basic_text_features": [
            "num_pronouns",
            "num_nouns",
            "num_verbs",
            "num_adjs",
            "num_advs",
            "num_chars",
            "num_tokens",
            "avg_token_len",
        ],
        "question_exclaim_features": [
            "ends_with_qmark",
            "ends_with_exclaim",
            "has_quote",
            "has_all_caps_word",
        ],
        "sentence_style": [
            "headline_imperative_verb",
            "headline_has_curiosity_word",
            "headline_curiosity_similarity",
            "headline_has_intensity_word",
            "headline_intensity_similarity",
        ],
        "sentiment": [
            "neg",
            "neu",
            "pos",
            "compound",
        ],
        "test_size": [
            "test_group_size",
        ],
        "readability_specificity": [
            "read_flesch",
            "read_coleman",
            "specificity_tfidf",
        ],
    }

    group_models = {}

    for group_name, cols in feature_groups.items():
        print("\n")
        print(f"Group: {group_name} — features: {cols}")
        print("=================\n")

        # Subset data and drop rows with missing values
        model_df = df[cols + [target_col]].dropna()

        X = model_df[cols].astype(float)
        y = model_df[target_col].astype(float)

        X = sm.add_constant(X)

        model = sm.OLS(y, X).fit()
        group_models[group_name] = model

        print(model.summary())

feature_groups_regression(exploratory_df)



Group: named_entities — features: ['headline_num_persons', 'headline_num_orgs', 'headline_num_gpes']

                            OLS Regression Results                            
Dep. Variable:                    ctr   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     60.65
Date:                Sat, 06 Dec 2025   Prob (F-statistic):           4.78e-39
Time:                        23:47:19   Log-Likelihood:                 67575.
No. Observations:               22644   AIC:                        -1.351e+05
Df Residuals:                   22640   BIC:                        -1.351e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-----------------

**Named entities**

The mention of people clearly increases the CTR while the mention of money decreases it.

**Sentence starting**

Headlines starting with a pronoun are associated with a lower CTR, this isn't something we expected.

**Basic text features**

The only conclusion we can draw from this is that users prefer shorter headlines in general.

**Question/Exclamation features**

Headlines with questions, exclamation marks, or all caps words have clearly a lower CTR.

**Sentence style**

Not much can be interpreted from this, statistically headlines are not affected by the use of curiosity or intensity words, as well as the use of verbs in imperative modes. I expected something different from this. I would have assumed that the use of curiosity words would increase the CTR.

**Sentiment**

Compound ranges from -1 to 1, with 0 being neutral, -1 negative and 1 positive. Positiveness is associated with a decrease in the CTR.

**Test Size**

It can't be denied that the use of multiple headlines (packages) in an A/B test decreases the CTR overall. This may be because the test of more headlines, include more versions of the headlines that aren't so effective.

**Specificity**

The use of specific words is associated with a decrease in the CTR.

In general, the regression of the features independently gave us the same results as the evaluation of all the features simultaneously.

## Hypothesis Confirmation

Now I'll run the same model on the confirmatory dataset to evaluate if the same correlations hold.

In [20]:
feature_groups_regression(confirmatory_df)



Group: named_entities — features: ['headline_num_persons', 'headline_num_orgs', 'headline_num_gpes']

                            OLS Regression Results                            
Dep. Variable:                    ctr   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     141.2
Date:                Sun, 07 Dec 2025   Prob (F-statistic):           2.62e-91
Time:                        00:00:24   Log-Likelihood:             3.1358e+05
No. Observations:              105420   AIC:                        -6.272e+05
Df Residuals:                  105416   BIC:                        -6.271e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-----------------

- **Named entities**: Valid for the confirmatory dataset too
- **Sentence starting**: Valid for the confirmatory dataset too
- **Basic text features**: Valid for the confirmatory dataset too
- **Question/Exclamation features**: Valid for the confirmatory dataset too
- **Sentence style**: There is a slight decrease in the CTR for headlines with words related to curiosity
- **Sentiment**: Here the correlation is not statistically significant
- **Test Size**: Valid for the confirmatory dataset too
- **Readability and specificity**: Valid for the confirmatory dataset too